In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [3]:
Payments = pd.read_csv('data/Internal/installments_payments.csv')

In [4]:
Payments.shape

(13605401, 8)

In [5]:
Payments.dtypes

SK_ID_PREV                  int64
SK_ID_CURR                  int64
NUM_INSTALMENT_VERSION    float64
NUM_INSTALMENT_NUMBER       int64
DAYS_INSTALMENT           float64
DAYS_ENTRY_PAYMENT        float64
AMT_INSTALMENT            float64
AMT_PAYMENT               float64
dtype: object

In [6]:
Payments = Payments.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

In [7]:
Payments.sort_values(by=['SK_ID_CURR', 'SK_ID_PREV', 'NUM_INSTALMENT_NUMBER'], inplace=True)

Check if `NUM_INSTALMENT_VERSION` keeps changing or not.

In [8]:
Payments['diff'] = Payments.groupby(['SK_ID_PREV'])['NUM_INSTALMENT_VERSION'].diff()

In [9]:
def get_payment_freq(x):
    approved = full_df.loc[x.index, 'NAME_CONTRACT_STATUS'] == 'Approved'
    return x[approved].iloc[0] if approved.any() else np.nan

In [10]:
Payments['LATE_PAYMENT_DAYS'] = Payments['DAYS_ENTRY_PAYMENT'] - Payments['DAYS_INSTALMENT']

Payments['INSTALLMENT_RATIO'] = np.where(Payments['AMT_PAYMENT'] > 0 ,
                                         Payments['AMT_INSTALMENT'] / Payments['AMT_PAYMENT'] ,
                                         np.nan)
Payments['LATE_PAYMENT_DAYS_LASTYEAR'] = np.where(Payments['DAYS_ENTRY_PAYMENT'] >= -365,
                                                 Payments['LATE_PAYMENT_DAYS'],
                                                 np.nan)

In [11]:
Payments_Silver = Payments.groupby(['SK_ID_CURR', 'SK_ID_PREV']).agg(
    PMNT_INSTLMT_VRSN_CHANGES_COUNT = ('NUM_INSTALMENT_VERSION', lambda x: x.nunique() ),
    PMNT_INSTLMT_DURATION_DAYS = ('DAYS_ENTRY_PAYMENT', lambda x: x.max() - x.min() + 1),
    PMNT_AVG_INSTALLMENT_FREQ = ('DAYS_ENTRY_PAYMENT', lambda x: pd.Series(x.unique()).sort_values().diff().mean() if len(x) > 1 else np.nan),
    PMNT_AVG_LATE_DAYS = ('LATE_PAYMENT_DAYS', lambda x: x.mean() if len(x) > 0 else np.nan),
    PMNT_AVG_LATE_DAYS_LASTYEAR = ('LATE_PAYMENT_DAYS_LASTYEAR', lambda x: x.notna().mean()),
    PMNT_AVG_INSTALLMENT_RATIO = ('INSTALLMENT_RATIO', lambda x: x.mean() if len(x) > 0 else np.nan),
    PMNT_NUM_INSTALMENTS = ('NUM_INSTALMENT_NUMBER', 'count'),
    PMNT_NUM_INSTALMENTS_LASTYEAR = ('DAYS_ENTRY_PAYMENT', lambda x:  x[x >= -365].count() if len(x) > 0 else np.nan)
).reset_index()

`PMNT_AVG_INSTALLMENT_FREQ` the average days between payments, is negative in few cases. This is because in some IDs, the `DAYS_ENTRY_PAYMENT` is decreasng with Installment Number , ie higher installment number indicates an earlier payment rather than a later payment. `NUM_INSTALMENT_NUMBER` is not in correct order, need to sort data based on `DAYS_ENTRY_PAYMENT`.

In [12]:
Payments_Gold = Payments_Silver.groupby(['SK_ID_CURR']).agg(
    PMNT_INSTLMT_VRSN_AVG_CHANGES = ('PMNT_INSTLMT_VRSN_CHANGES_COUNT', 'mean'),
    PMNT_INSTLMT_VRSN_MAX_CHANGES = ('PMNT_INSTLMT_VRSN_CHANGES_COUNT', 'max'),
    PMNT_INSTLMT_VRSN_MEDIAN_CHANGES = ('PMNT_INSTLMT_VRSN_CHANGES_COUNT', 'median'),

    PMNT_INSTLMT_DURATION_DAYS = ('PMNT_INSTLMT_DURATION_DAYS', 'mean'),
    PMNT_INSTLMT_DURATION_MAX_DAYS = ('PMNT_INSTLMT_DURATION_DAYS', 'max'),
    PMNT_INSTLMT_DURATION_MEDIAN_DAYS = ('PMNT_INSTLMT_DURATION_DAYS', 'median'),

    PMNT_AVG_AVG_INSTALLMENT_FREQ = ('PMNT_AVG_INSTALLMENT_FREQ', 'mean'),
    PMNT_MAX_AVG_INSTALLMENT_FREQ = ('PMNT_AVG_INSTALLMENT_FREQ', 'max'),
    PMNT_MEDIAN_AVG_INSTALLMENT_FREQ = ('PMNT_AVG_INSTALLMENT_FREQ', 'median'),

    PMNT_AVG_LATE_DAYS = ('PMNT_AVG_LATE_DAYS', 'mean'),
    PMNT_MAX_LATE_DAYS = ('PMNT_AVG_LATE_DAYS', 'max'),
    PMNT_MEDIAN_LATE_DAYS = ('PMNT_AVG_LATE_DAYS', 'median'),

    PMNT_AVG_AVG_LATE_DAYS_LASTYEAR = ('PMNT_AVG_LATE_DAYS_LASTYEAR', 'mean'),
    PMNT_MAX_AVG_LATE_DAYS_LASTYEAR = ('PMNT_AVG_LATE_DAYS_LASTYEAR', 'max'),
    PMNT_MEDIAN_AVG_LATE_DAYS_LASTYEAR = ('PMNT_AVG_LATE_DAYS_LASTYEAR', 'median'),

    PMNT_AVG_INSTALLMENT_RATIO = ('PMNT_AVG_INSTALLMENT_RATIO', 'mean'),
    PMNT_MEDIAN_INSTALLMENT_RATIO = ('PMNT_AVG_INSTALLMENT_RATIO', 'median'),

    PMNT_AVG_NUM_INSTALMENTS = ('PMNT_NUM_INSTALMENTS', 'mean'),
    PMNT_MAX_NUM_INSTALMENTS = ('PMNT_NUM_INSTALMENTS', 'max'),
    PMNT_MEDIAN_NUM_INSTALMENTS = ('PMNT_NUM_INSTALMENTS', 'median'),

    PMNT_AVG_NUM_INSTALMENTS_LASTYEAR = ('PMNT_NUM_INSTALMENTS_LASTYEAR', 'mean'),
    PMNT_MAX_NUM_INSTALMENTS_LASTYEAR = ('PMNT_NUM_INSTALMENTS_LASTYEAR', 'max'),
    PMNT_MEDIAN_NUM_INSTALMENTS_LASTYEAR = ('PMNT_NUM_INSTALMENTS_LASTYEAR', 'median')
).reset_index()

In [13]:
Payments_Gold.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\installments_processed.parquet")